<a href="https://colab.research.google.com/github/JananiGovuri/Natural-Language-Processing-/blob/main/HW4_Code_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# ---- Sample Toy Data ----
text = "hello help hello hero hello hill hello help hello"
chars = sorted(list(set(text)))

# Create mappings
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for ch, i in char2idx.items()}

vocab_size = len(chars)

# Encode text
data = [char2idx[ch] for ch in text]

# ---- Hyperparameters ----
hidden_size = 128
seq_length = 20
learning_rate = 0.003
epochs = 10

# ---- Model Definition ----
class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(CharRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

# Initialize model
model = CharRNN(vocab_size, hidden_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# ---- Training ----
for epoch in range(epochs):
    total_loss = 0
    hidden = None

    for i in range(0, len(data) - seq_length):
        seq_in = data[i:i+seq_length]
        seq_out = data[i+1:i+seq_length+1]

        inputs = torch.tensor(seq_in).unsqueeze(0)
        targets = torch.tensor(seq_out)

        optimizer.zero_grad()

        outputs, hidden = model(inputs, hidden)
        hidden = (hidden[0].detach(), hidden[1].detach())

        loss = criterion(outputs.squeeze(), targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# ---- Text Generation ----
def generate_text(model, start_str, length=100, temperature=1.0):
    model.eval()
    input_seq = torch.tensor([char2idx[ch] for ch in start_str]).unsqueeze(0)
    hidden = None

    result = start_str

    for _ in range(length):
        output, hidden = model(input_seq, hidden)
        logits = output[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=1).detach().numpy()[0]

        next_char = random.choices(range(vocab_size), weights=probs)[0]
        result += idx2char[next_char]

        input_seq = torch.tensor([[next_char]])

    return result

# ---- Generate Samples ----
print("\nSample (τ=0.7):")
print(generate_text(model, "he", 100, temperature=0.7))

print("\nSample (τ=1.0):")
print(generate_text(model, "he", 100, temperature=1.0))

print("\nSample (τ=1.2):")
print(generate_text(model, "he", 100, temperature=1.2))

Epoch 1, Loss: 21.3934
Epoch 2, Loss: 9.9590
Epoch 3, Loss: 9.0197
Epoch 4, Loss: 8.1038
Epoch 5, Loss: 6.9257
Epoch 6, Loss: 6.2002
Epoch 7, Loss: 5.9885
Epoch 8, Loss: 5.4771
Epoch 9, Loss: 5.1310
Epoch 10, Loss: 5.0510

Sample (τ=0.7):
hero hello hill hello hill hello hero hello hill hello hero hello help hello hill hello hero hello hil

Sample (τ=1.0):
hello hero hello hello help hello hill hello hill hello hill hill hello hill hello hill hello hill hel

Sample (τ=1.2):
help hill hello help hello hello ho hello hill hello hero hello hill hello hill hello help hello hill 
